# Python para Mapeamento Urbano
### Live coding: do dado público ao mapa

**Oficina para estudantes de Arquitetura e Urbanismo** · Felipe Almeida ([felipe-almeida.com](https://felipe-almeida.com/))

Este notebook percorre, em ordem, tudo o que vimos nos slides. Cada seção começa com uma pergunta urbana e termina com um produto (tabela, mapa ou gráfico). Rode as células de cima para baixo com `Shift + Enter`.

| Seção | Pergunta | Ferramenta |
|---|---|---|
| 1 | Como o Python funciona? | `print`, listas, dicionários, funções, loops, `pandas` |
| 2–4 | Quantas pessoas vivem em cada município de MG? | API SIDRA, `merge` |
| 5 | Como medir área e distância sem errar? | Projeções (CRS) |
| 6 | A densidade cai quando nos afastamos de BH? | Correlação e regressão |
| 7–8 | Como é a densidade dentro de BH, bairro a bairro? | Setores censitários, mapas em loop |
| 9 | Quem mora perto do metrô? | Buffer e join espacial |
| 10 | Onde se concentram bares e restaurantes? | Cluster (DBSCAN) |
| 11 | Como é o tecido do Centro? Qual o caminho a pé? | OpenStreetMap (`osmnx`) |
| 12 | Quais nós seguram o sistema de transporte da RMBH? | Grafos (`networkx`) |
| 13 | Como compartilhar o resultado? | Mapa interativo e painel |

Os exercícios do fim da oficina estão em `02_exercicios.ipynb`.

> Todo mapa e gráfico gerado aqui leva o crédito de autoria automaticamente pela função `creditos()`.

## 0. Preparar o ambiente

O Colab já vem com `pandas`, `geopandas`, `matplotlib` e `networkx`. Instalamos o que falta e trazemos o repositório da oficina (que tem as funções de estilo e de carregamento de dados).

In [ ]:
# Rode uma vez por sessão (leva ~1 minuto)
!pip install -q geobr osmnx mapclassify folium

import os, sys
REPO = "https://github.com/brfelipealmeida/oficina-python-mapeamento"
if not os.path.exists("oficina-python-mapeamento"):
    !git clone -q {REPO}
sys.path.append("oficina-python-mapeamento/src")

In [ ]:
import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import networkx as nx

from estilo import *   # cores, fontes, créditos, barra de escala
from dados import *    # funções que baixam IBGE, geobr e OSM
from redes import *    # funções prontas de grafos
from painel import exportar_painel

configurar_estilo()
pd.set_option("display.max_columns", 30)

## 1. Python em 15 minutos

Antes dos mapas, o mínimo de Python para ler todo o resto do notebook. Cada célula é independente: rode, leia o resultado e mude alguma coisa para ver o que acontece.

| Ideia | Exemplo | Para que serve na oficina |
|---|---|---|
| `print` | `print("Hello, World!")` | mostrar resultados |
| variável | `pop = 2315560` | guardar um valor com nome |
| lista | `[1, 2, 3]` | uma sequência (estações, bairros) |
| dicionário | `{"chave": valor}` | pares nome → valor (como uma ficha) |
| função | `def densidade(p, a):` | repetir um cálculo sem reescrever |
| loop | `for b in bairros:` | fazer a mesma coisa para cada item |
| biblioteca | `import pandas as pd` | usar ferramentas que outras pessoas escreveram |

### 1.1 O famoso Hello World

In [ ]:
print("Hello, World!")
print("Olá, Belo Horizonte!")

### 1.2 Python como calculadora

In [ ]:
print(10 + 5)     # soma
print(10 - 5)     # subtração
print(10 * 5)     # multiplicação
print(10 / 4)     # divisão (sempre dá decimal)
print(10 // 4)    # divisão inteira
print(10 % 4)     # resto da divisão
print(2 ** 10)    # potência

### 1.3 Variáveis e tipos

Uma variável é um nome para um valor. O tipo (`type`) diz o que o valor é: número inteiro (`int`), decimal (`float`), texto (`str`) ou verdadeiro/falso (`bool`). Esses mesmos tipos voltam depois como colunas de tabelas.

In [ ]:
municipio = "Belo Horizonte"   # str
populacao = 2315560            # int (Censo 2022)
area_km2 = 331.4               # float (aproximada)
capital = True                 # bool

densidade = populacao / area_km2
print(type(municipio), type(populacao), type(area_km2), type(capital))
print(f"{municipio} tem {densidade:,.0f} habitantes por km²")

### 1.4 Listas: uma sequência de coisas

A contagem começa do **zero**: o primeiro item é `[0]`, o último é `[-1]`.

In [ ]:
estacoes = ["Eldorado", "Lagoinha", "Central", "Santa Tereza", "Vilarinho"]

print(estacoes[0])        # primeiro
print(estacoes[-1])       # último
print(estacoes[1:3])      # do segundo ao terceiro
print(len(estacoes))      # quantos itens

estacoes.append("São Gabriel")
print(estacoes)

### 1.5 Dicionários: pares de nome e valor

Um dicionário funciona como uma ficha: cada **chave** aponta para um **valor**. É a mesma ideia de uma linha de tabela.

In [ ]:
estacao = {
    "nome": "Lagoinha",
    "linha": 1,
    "integracao": ["Linha 3 (projeto)", "VLT (projeto)"],
    "lat": -19.9129,
    "lon": -43.9425,
}
print(estacao["nome"])
print(estacao["integracao"])
print(estacao.keys())

estacao["bairro"] = "Lagoinha"   # acrescenta uma chave
print(estacao)

### 1.6 Condições e loops

`if` decide; `for` repete. A indentação (os espaços no começo da linha) faz parte da linguagem: é ela que diz o que está "dentro" do loop.

In [ ]:
densidades = {"Centro": 120.5, "Savassi": 95.2, "Barreiro": 48.7, "Venda Nova": 61.3}

for bairro, d in densidades.items():
    if d > 80:
        print(f"{bairro}: denso ({d} hab/ha)")
    else:
        print(f"{bairro}: menos denso ({d} hab/ha)")

### 1.7 Funções: escrever uma vez, usar sempre

In [ ]:
def densidade_hab_ha(pessoas, area_m2):
    """Pessoas por hectare (1 ha = 10.000 m²)."""
    return pessoas / (area_m2 / 10_000)

print(densidade_hab_ha(1200, 40_000))
print(densidade_hab_ha(300, 90_000))

### 1.8 Bibliotecas: ferramentas prontas

Python sozinho é só a linguagem. A força está nas bibliotecas:

| Biblioteca | Faz o quê | Apelido usual |
|---|---|---|
| `pandas` | tabelas (DataFrame) | `pd` |
| `geopandas` | tabelas com geometria | `gpd` |
| `matplotlib` | gráficos e mapas estáticos | `plt` |
| `numpy` | contas com muitos números | `np` |
| `osmnx` | dados do OpenStreetMap | `ox` |
| `networkx` | grafos e redes | `nx` |

Elas já foram importadas na seção 0 com `import pandas as pd` etc.

### 1.9 Carregar um dado

Uma tabela pode vir de um arquivo (`.csv`, `.xlsx`, `.gpkg`) ou de um endereço na internet. O resultado é um **DataFrame**.

In [ ]:
metro = pd.read_csv("oficina-python-mapeamento/dados/metro_bh_linha1.csv")
print(metro.shape)      # (linhas, colunas)
print(metro.columns)    # nomes das colunas
metro.head()            # cinco primeiras linhas

In [ ]:
# Selecionar colunas, filtrar linhas, ordenar
print(metro["estacao"].head(3))
metro[metro["lat"] > -19.88].sort_values("lat", ascending=False)

Pronto: com variáveis, listas, dicionários, loops, funções e DataFrames dá para ler tudo o que vem a seguir.

## 2. Dado não espacial: população 2022 pela API do IBGE

Uma **API** é um endereço na internet que devolve dados em vez de páginas. A SIDRA é a API de tabelas do IBGE. Aqui pedimos a **tabela 4714** (população residente do Censo 2022) para todos os municípios de Minas Gerais (código de UF 31).

Esse dado é **tabular**: tem código, nome e número. Não tem geometria, então ainda não dá para mapear.

In [ ]:
pop = populacao_municipios_sidra(uf=31)
pop.head()

### Tipos de coluna: por que importam?

Todo `DataFrame` tem um tipo por coluna (`dtype`). Os principais:

| dtype | O que guarda | Exemplo urbano |
|---|---|---|
| `object` / `string` | texto | nome do bairro, **código IBGE** |
| `int64` | inteiro | número de domicílios |
| `float64` | decimal | densidade, área |
| `bool` | verdadeiro/falso | está em favela? |
| `geometry` | ponto, linha, polígono | lote, rua, estação |

Códigos (IBGE, CEP, setor censitário) **são texto**, mesmo sendo feitos de dígitos: não se soma código, e zeros à esquerda importam.

In [ ]:
pop.dtypes

In [ ]:
# A população veio como texto (a API devolve JSON). Convertemos:
pop["populacao"] = pd.to_numeric(pop["populacao"], errors="coerce")
pop["cod_mun"] = pop["cod_mun"].astype(str)
pop.dtypes

### Validação: antes de analisar, desconfie

Perguntas mínimas para qualquer base: quantas linhas deveria ter? há duplicados? há vazios? os extremos fazem sentido?

In [ ]:
print("Municípios:", len(pop), "(MG tem 853)")
print("Códigos duplicados:", pop["cod_mun"].duplicated().sum())
print("População vazia:", pop["populacao"].isna().sum())
print("Total MG:", f'{pop["populacao"].sum():,.0f}')
pop.nlargest(5, "populacao")

## 3. Dado espacial: limites municipais

O pacote `geobr` (IPEA) entrega as malhas oficiais do IBGE prontas para o Python. O resultado é um **GeoDataFrame**: uma tabela com uma coluna especial, `geometry`.

In [ ]:
mun = municipios_mg()
print(mun.crs)
mun.head()

In [ ]:
print(mun.dtypes)
print(mun.geom_type.value_counts())

## 4. O join: tabela + geometria

O **join por atributo** (`merge`) junta duas tabelas por uma coluna em comum. Aqui, o código do município.

Primeiro, o erro clássico: tentar juntar colunas de tipos diferentes.

In [ ]:
try:
    mun.merge(pop, left_on="code_muni", right_on="cod_mun")
except ValueError as erro:
    print("Erro:", erro)

In [ ]:
# O geobr guarda o código como número decimal (3106200.0). Convertemos para texto de 7 dígitos.
mun["cod_mun"] = mun["code_muni"].astype("int64").astype(str)

mg = mun.merge(pop, on="cod_mun", how="left", validate="one_to_one", indicator=True)
mg["nome"] = mg["municipio"].str.replace(" - MG", "", regex=False)
mg["_merge"].value_counts()

`both` em todas as linhas significa que nenhum município ficou sem população. Se aparecer `left_only`, investigue antes de seguir: geralmente é código com formato diferente ou município criado/extinto entre bases de anos distintos.

## 5. Projeções: a Terra não é uma prancha

O IBGE publica em **SIRGAS 2000 geográfico (EPSG:4674)**: coordenadas em graus. Graus servem para localizar, não para medir. Para área e distância, projetamos:

| CRS | Unidade | Quando usar |
|---|---|---|
| EPSG:4674 (SIRGAS 2000) | graus | guardar e trocar dados oficiais |
| EPSG:4326 (WGS84) | graus | OpenStreetMap, GPS, mapas web |
| **EPSG:31983 (SIRGAS 2000 / UTM 23S)** | metros | **análises em BH e entorno** |
| ESRI:102033 (Albers, América do Sul) | metros | áreas em escala estadual (preserva área) |

Pense numa planta: antes de medir, você precisa saber a escala.

In [ ]:
# Área em "graus quadrados" não significa nada:
print("Área de BH em graus²:", mg.loc[mg.cod_mun == COD_BH].area.values)

# Em projeção de área igual:
mg_albers = mg.to_crs("ESRI:102033")
mg["area_km2"] = mg_albers.area / 1e6
mg["densidade"] = mg["populacao"] / mg["area_km2"]
mg.loc[mg.cod_mun == COD_BH, ["nome", "populacao", "area_km2", "densidade"]]

**Validação cruzada:** o IBGE informa cerca de 331 km² para BH (confira no IBGE Cidades). Se o seu número estiver muito diferente, a projeção ou a geometria têm problema.

In [ ]:
fig, ax = mapa_base((9, 9))
mg.plot(column="densidade", scheme="quantiles", k=5, cmap=CMAP,
        linewidth=0.1, edgecolor="white", legend=True, ax=ax,
        legend_kwds={"title": "hab/km²", "fmt": "{:.0f}", "loc": "upper left"})
titulo(ax, "Densidade demográfica em Minas Gerais", "Municípios, 2022 · classes por quintis")
creditos(fig, fonte="IBGE, Censo 2022; geobr/IPEA")
salvar(fig, "01_densidade_mg")

## 6. Do mapa à pergunta: hipótese, correlação e regressão

Um mapa bonito não é uma conclusão. O caminho é:

1. **Pergunta**: a densidade está ligada à proximidade de BH?
2. **Hipótese (H1)**: quanto mais longe de BH, menor a densidade. **H0**: não há relação.
3. **Variáveis**: dependente (o que quero explicar) = densidade; independente = distância a BH.
4. **Teste**: correlação (força e direção) e regressão (quanto muda).
5. **Interpretação e limites**: correlação não é causalidade.

Essa ideia de gradiente de densidade a partir do centro é clássica na economia urbana (Clark, 1951).

In [ ]:
# Distância do centróide de cada município ao centróide de BH (em metros, projeção métrica)
centro_bh = mg_albers.loc[mg_albers.cod_mun == COD_BH].geometry.centroid.iloc[0]
mg["dist_bh_km"] = mg_albers.geometry.centroid.distance(centro_bh) / 1000

amostra = mg[mg.cod_mun != COD_BH].copy()   # BH tem distância 0: log(0) não existe
amostra[["densidade", "dist_bh_km"]].describe().round(1)

In [ ]:
# Pearson mede relação linear; Spearman mede relação monotônica (por ranking).
print("Pearson  (bruto):", round(amostra["densidade"].corr(amostra["dist_bh_km"]), 3))
print("Spearman (bruto):", round(amostra["densidade"].corr(amostra["dist_bh_km"], method="spearman"), 3))
amostra["log_dens"] = np.log(amostra["densidade"])
amostra["log_dist"] = np.log(amostra["dist_bh_km"])
print("Pearson  (log-log):", round(amostra["log_dens"].corr(amostra["log_dist"]), 3))

A densidade é muito **assimétrica** (poucos municípios muito densos, muitos pouco densos). Por isso trabalhamos em logaritmo: a relação fica mais próxima de uma reta e os extremos pesam menos.

In [ ]:
import statsmodels.formula.api as smf

modelo = smf.ols("log_dens ~ log_dist", data=amostra).fit()
print(modelo.summary().tables[1])
print(f"R²: {modelo.rsquared:.2f}")
b = modelo.params["log_dist"]
print(f"Leitura: 1% mais longe de BH está associado a {b:.2f}% na densidade.")

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5.5))
ax.scatter(amostra["dist_bh_km"], amostra["densidade"], s=12, alpha=0.6, color=CORES["lilas"])
x = np.linspace(amostra["dist_bh_km"].min(), amostra["dist_bh_km"].max(), 200)
ax.plot(x, np.exp(modelo.params["Intercept"]) * x ** b, color=CORES["roxo"], lw=2)
for nome in ["Contagem", "Juiz de Fora", "Uberlândia", "Montes Claros", "Ipatinga"]:
    r = amostra[amostra["nome"] == nome]
    if not r.empty:
        ax.annotate(nome, (r["dist_bh_km"].iloc[0], r["densidade"].iloc[0]), fontsize=8,
                    xytext=(4, 4), textcoords="offset points")
ax.set_xscale("log"); ax.set_yscale("log")
ax.set_xlabel("Distância a BH (km, escala log)"); ax.set_ylabel("Densidade (hab/km², escala log)")
titulo(ax, "Densidade cai com a distância a BH?", f"Regressão log-log · R² = {modelo.rsquared:.2f}")
creditos(fig, fonte="IBGE, Censo 2022; geobr/IPEA")
salvar(fig, "02_regressao_densidade_distancia")

### Os resíduos também são um mapa

Resíduo = valor observado − valor previsto. Se os resíduos formam manchas no mapa, falta uma variável espacial no modelo (por exemplo, outros polos regionais além de BH). Isso se chama **autocorrelação espacial**, e é o começo de outra conversa.

In [ ]:
amostra["residuo"] = modelo.resid
lim = amostra["residuo"].abs().quantile(0.95)
fig, ax = mapa_base((9, 9))
mg.plot(ax=ax, color=CORES["cinza_claro"], linewidth=0)
amostra.plot(column="residuo", cmap=CMAP_DIV, vmin=-lim, vmax=lim, linewidth=0.1,
             edgecolor="white", ax=ax, legend=True,
             legend_kwds={"shrink": 0.5, "label": "mais denso que o previsto →"})
titulo(ax, "Onde o modelo erra", "Resíduos da regressão densidade ~ distância a BH")
creditos(fig, fonte="IBGE, Censo 2022; geobr/IPEA")
salvar(fig, "03_residuos")

## 7. Mudando de escala: setores censitários de BH

O **setor censitário** é a menor unidade de divulgação do Censo (algumas quadras). O IBGE publica a malha de 2022 **já com atributos** (pessoas, domicílios). Carregamos BH e Contagem, porque a estação Eldorado fica em Contagem.

> O arquivo de MG tem ~180 MB e é baixado uma vez só.

In [ ]:
COD_CONTAGEM = "3118601"
setores = pd.concat([setores_bh(COD_BH), setores_bh(COD_CONTAGEM)], ignore_index=True)
print(setores.shape, setores.crs)
setores[["CD_SETOR", "NM_MUN", "NM_BAIRRO", "pessoas", "domicilios", "em_favela"]].head()

In [ ]:
# Validação cruzada: a soma dos setores bate com a SIDRA?
soma = setores.groupby("CD_MUN")["pessoas"].sum()
ref = pop.set_index("cod_mun")["populacao"]
pd.DataFrame({"soma_setores": soma, "sidra": ref.reindex(soma.index.astype(str)).values})

In [ ]:
setores["area_ha"] = setores.area / 10_000          # CRS em metros: área em m² → ha
setores["dens_ha"] = setores["pessoas"] / setores["area_ha"]
print("Setores sem população:", (setores["pessoas"].fillna(0) == 0).sum())

bh = setores[setores.CD_MUN == COD_BH]
fig, ax = mapa_base((9, 10))
bh.plot(column="dens_ha", scheme="quantiles", k=6, cmap=CMAP, linewidth=0, ax=ax,
        legend=True, missing_kwds={"color": CORES["cinza_claro"]},
        legend_kwds={"title": "hab/ha", "fmt": "{:.0f}", "loc": "lower right"})
barra_escala(ax, 2000); seta_norte(ax)
titulo(ax, "Densidade em Belo Horizonte", "Setores censitários, 2022 · quantis")
creditos(fig, fonte="IBGE, Censo 2022, Malha de Setores com atributos")
salvar(fig, "04_densidade_setores_bh")

## 8. Mapas em série: um mapa por bairro

Aqui o Python mostra sua maior vantagem sobre fazer mapas "na mão": **o mesmo mapa para cada bairro**, com a mesma escala de cores, em segundos. Primeiro uma prancha com vários bairros lado a lado (*small multiples*), depois um arquivo para cada bairro.

Escolhemos automaticamente os nove bairros mais populosos. Troque `bairros` por uma lista sua, por exemplo `["Centro", "Savassi", "Lourdes"]`.

In [ ]:
bairros = bh.groupby("NM_BAIRRO")["pessoas"].sum().nlargest(9).index.tolist()
print(bairros)

# a mesma escala de cores para todos: assim os mapas são comparáveis
vmax = bh["dens_ha"].quantile(0.95)

In [ ]:
fig, axs = plt.subplots(3, 3, figsize=(12, 12))
for ax, nome in zip(axs.flat, bairros):
    recorte = bh[bh["NM_BAIRRO"] == nome]
    recorte.plot(ax=ax, column="dens_ha", cmap=CMAP, vmin=0, vmax=vmax,
                 edgecolor="white", linewidth=0.3)
    ax.set_title(nome, loc="left", fontsize=13, fontweight="semibold")
    ax.set_axis_off()
    ax.set_aspect("equal")
fig.suptitle("Densidade nos nove bairros mais populosos de BH", x=0.02, ha="left",
             fontsize=22, fontweight="semibold", fontfamily="Barlow Condensed")
creditos(fig, fonte="IBGE, Censo 2022", nota="mesma escala de cores em todos os mapas (0 a p95)")
salvar(fig, "bairros_small_multiples")

Agora um arquivo por bairro. O `for` repete o bloco inteiro; só o nome do bairro muda. Os arquivos ficam em `outputs/bairros/`.

In [ ]:
import os
os.makedirs("outputs/bairros", exist_ok=True)

for nome in bairros:
    recorte = bh[bh["NM_BAIRRO"] == nome]
    fig, ax = mapa_base((7, 7))
    recorte.plot(ax=ax, column="dens_ha", cmap=CMAP, vmin=0, vmax=vmax,
                 edgecolor="white", linewidth=0.4, legend=True,
                 legend_kwds={"shrink": 0.6, "label": "hab/ha"})
    barra_escala(ax, 250)
    titulo(ax, nome, f"{recorte['pessoas'].sum():,.0f} moradores · {len(recorte)} setores")
    creditos(fig, fonte="IBGE, Censo 2022")
    arquivo = nome.lower().replace(" ", "_")
    salvar(fig, arquivo, pasta="outputs/bairros")
    plt.close(fig)   # fecha a figura para não acumular memória

## 9. Buffer e join espacial: quem mora perto do metrô?

- **Buffer**: área a uma distância fixa de algo. 800 m ≈ 10 minutos a pé, referência comum em planejamento orientado ao transporte (TOD).
- **Join espacial** (`sjoin`): junta por relação geométrica (dentro, intersecta, mais próximo), não por código.

Duas formas de contar a população do buffer, e elas dão resultados diferentes:

In [ ]:
est = metro_bh()
buf = est[["estacao", "geometry"]].copy()
buf["geometry"] = est.buffer(800)

# A) pelo centróide: o setor inteiro entra se o centro dele estiver no buffer
cent = setores[["pessoas", "geometry"]].copy()
cent["geometry"] = setores.centroid
a = gpd.sjoin(cent, buf, predicate="within")
pop_centroide = a.groupby("estacao")["pessoas"].sum()

# B) proporcional à área: só a fração do setor dentro do buffer
setores["area_orig"] = setores.area
inter = gpd.overlay(setores[["pessoas", "area_orig", "geometry"]], buf, how="intersection")
inter["pessoas_prop"] = inter["pessoas"] * inter.area / inter["area_orig"]
pop_proporcional = inter.groupby("estacao")["pessoas_prop"].sum()

comp = pd.DataFrame({"centroide": pop_centroide, "proporcional": pop_proporcional}).round(0)
comp.reindex(est["estacao"])

**Atenção:** os buffers de estações vizinhas se sobrepõem, então a mesma pessoa conta para duas estações. Para o total da linha, dissolvemos os buffers antes de somar.

In [ ]:
area_metro = buf.dissolve()
dentro = gpd.overlay(setores[["pessoas", "area_orig", "CD_MUN", "geometry"]], area_metro, how="intersection")
dentro["pessoas_prop"] = dentro["pessoas"] * dentro.area / dentro["area_orig"]
pct = dentro.loc[dentro.CD_MUN == COD_BH, "pessoas_prop"].sum() / bh["pessoas"].sum()
print(f"{pct:.1%} da população de BH mora a até 800 m de uma estação da Linha 1.")

In [ ]:
fig, ax = mapa_base((9, 9))
setores.plot(column="dens_ha", scheme="quantiles", k=6, cmap=CMAP, linewidth=0, ax=ax, alpha=0.9)
area_metro.boundary.plot(ax=ax, color=CORES["minerio"], linewidth=1.5)
est.plot(ax=ax, color=CORES["tinta"], markersize=10, zorder=3)
ax.set_xlim(area_metro.total_bounds[[0, 2]] + [-2500, 2500])
ax.set_ylim(area_metro.total_bounds[[1, 3]] + [-2500, 2500])
barra_escala(ax, 2000); seta_norte(ax)
titulo(ax, "800 m da Linha 1", "Densidade por setor e área de influência das estações")
creditos(fig, fonte="IBGE, Censo 2022; posição das estações a validar com OSM")
salvar(fig, "05_buffer_metro")

## 10. Cluster espacial: onde se concentram bares, cafés e restaurantes?

O **DBSCAN** agrupa pontos que têm muitos vizinhos próximos. Dois parâmetros:
- `eps`: raio de vizinhança, **em metros** (por isso precisamos de CRS projetado);
- `min_samples`: mínimo de pontos para formar um núcleo.

Pontos sem grupo são **ruído** (`-1`). O resultado aponta centralidades de comércio e serviço.

In [ ]:
from sklearn.cluster import DBSCAN

pois = osm_feicoes({"amenity": ["restaurant", "bar", "cafe", "pub"]},
                   lugar="Belo Horizonte, Minas Gerais, Brazil")
pois["geometry"] = pois.geometry.centroid
xy = np.column_stack([pois.geometry.x, pois.geometry.y])

pois["cluster"] = DBSCAN(eps=200, min_samples=15).fit_predict(xy)
print("Pontos:", len(pois), "| clusters:", pois["cluster"].max() + 1,
      "| ruído:", (pois["cluster"] == -1).sum())

In [ ]:
nucleos = pois[pois.cluster >= 0].dissolve("cluster").convex_hull.to_frame("geometry")
nucleos["n"] = pois[pois.cluster >= 0].groupby("cluster").size()

fig, ax = mapa_base((9, 10))
bh.plot(ax=ax, color=CORES["cinza_claro"], linewidth=0)
pois[pois.cluster == -1].plot(ax=ax, color=CORES["cinza"], markersize=1, alpha=0.5)
pois[pois.cluster >= 0].plot(ax=ax, column="cluster", cmap="felipe_seq", markersize=3)
nucleos.boundary.plot(ax=ax, color=CORES["roxo"], linewidth=1)
barra_escala(ax, 2000); seta_norte(ax)
titulo(ax, "Centralidades de bares, cafés e restaurantes", "DBSCAN · eps = 200 m · mín. 15 pontos")
creditos(fig, fonte="OpenStreetMap")
salvar(fig, "06_clusters_dbscan")

Experimente mudar `eps` para 100 e 400. O que acontece com o número de centralidades? Não existe parâmetro "certo": existe parâmetro justificado pela pergunta.

## 11. OpenStreetMap: o tecido da cidade

O `osmnx` baixa do OpenStreetMap qualquer feição por **tags** (`building`, `landuse`, `leisure`...) e a rede de ruas como **grafo**. Vamos trabalhar num raio de 1,2 km no Centro.

In [ ]:
PONTO = (-19.9255, -43.9380)   # entre a Praça Sete e a Praça da Liberdade (lat, lon)
RAIO = 1200

recorte = gpd.GeoSeries(gpd.points_from_xy([PONTO[1]], [PONTO[0]]), crs=CRS_WEB).to_crs(CRS_BH).buffer(RAIO)
edif = osm_feicoes({"building": True}, ponto=PONTO, raio=RAIO)
edif = edif.clip(recorte).explode(index_parts=False)
edif = edif[edif.geom_type == "Polygon"]
print(len(edif), "edificações")

### Cheios e vazios (mapa figura-fundo)

O mapa de Nolli (Roma, 1748) é o ancestral deste gráfico: o construído em preto, o espaço aberto em branco.

In [ ]:
taxa = edif.union_all().area / recorte.area.iloc[0]

fig, ax = mapa_base((9, 9))
edif.plot(ax=ax, color=CORES["tinta"], linewidth=0)
recorte.boundary.plot(ax=ax, color=CORES["cinza"], linewidth=0.8)
barra_escala(ax, 500); seta_norte(ax)
titulo(ax, "Cheios e vazios no Centro de BH", f"Raio de {RAIO} m · {taxa:.0%} da área ocupada por edificações")
creditos(fig, fonte="OpenStreetMap")
salvar(fig, "07_cheios_vazios")

In [ ]:
# Uso do solo e áreas verdes no mesmo recorte
uso = osm_feicoes({"landuse": True}, ponto=PONTO, raio=RAIO)
uso = uso.clip(recorte).explode(index_parts=False)
uso = uso[uso.geom_type == "Polygon"]
verde = osm_feicoes({"leisure": ["park", "garden"], "landuse": ["grass", "forest"],
                     "natural": ["wood", "scrub"]}, ponto=PONTO, raio=RAIO)
verde = verde.clip(recorte).explode(index_parts=False)
verde = verde[verde.geom_type == "Polygon"]
print(uso["landuse"].value_counts().head(8))

In [ ]:
paleta_uso = {"residential": CORES["lilas"], "commercial": CORES["roxo"], "retail": CORES["roxo"],
              "industrial": CORES["minerio"], "institutional": "#7A86A8", "religious": "#A9A3B5"}
fig, ax = mapa_base((9, 9))
for tipo, g in uso.groupby("landuse"):
    g.plot(ax=ax, color=paleta_uso.get(tipo, CORES["cinza_claro"]), linewidth=0, label=tipo)
verde.plot(ax=ax, color=CORES["verde"], linewidth=0, alpha=0.9)
edif.plot(ax=ax, facecolor="none", edgecolor="white", linewidth=0.15)
recorte.boundary.plot(ax=ax, color=CORES["cinza"], linewidth=0.8)
barra_escala(ax, 500); seta_norte(ax)
titulo(ax, "Uso do solo e áreas verdes", "Verde: parques, jardins, gramados e matas")
creditos(fig, fonte="OpenStreetMap", nota="lacunas no mapa = áreas sem tag de uso no OSM")
salvar(fig, "08_uso_do_solo")

**Validação:** o OSM é colaborativo. Áreas em cinza claro ou vazias não são "sem uso": são **sem registro**. Antes de concluir, compare com uma base oficial (por exemplo, o cadastro de uso do solo da Prefeitura).

### Caminho de A até B a pé

A rede de ruas do `osmnx` **já é um grafo do networkx**: esquinas são nós, trechos de rua são arestas. O caminho mínimo é o mesmo algoritmo (Dijkstra) que usaremos no metrô.

In [ ]:
import osmnx as ox

G_ruas = ox.graph_from_point(PONTO, dist=2000, network_type="walk")
print(type(G_ruas), G_ruas.number_of_nodes(), "nós,", G_ruas.number_of_edges(), "arestas")

A = (-19.9320, -43.9380)   # Praça da Liberdade
B = (-19.9230, -43.9437)   # Mercado Central
na = ox.distance.nearest_nodes(G_ruas, X=A[1], Y=A[0])
nb = ox.distance.nearest_nodes(G_ruas, X=B[1], Y=B[0])
rota = ox.shortest_path(G_ruas, na, nb, weight="length")
metros = nx.path_weight(G_ruas, rota, weight="length")
print(f"Rota: {metros:,.0f} m, cerca de {metros / 80:.0f} min a pé (80 m/min)")

In [ ]:
G_proj = ox.project_graph(G_ruas, to_crs=CRS_BH)
ruas = ox.graph_to_gdfs(G_proj, nodes=False)
rota_gdf = ox.routing.route_to_gdf(G_proj, rota)

fig, ax = mapa_base((9, 9))
ruas.plot(ax=ax, color=CORES["cinza_claro"], linewidth=0.6)
edif.plot(ax=ax, color="#C9CCD3", linewidth=0)
rota_gdf.plot(ax=ax, color=CORES["minerio"], linewidth=4)
ax.set_xlim(recorte.total_bounds[[0, 2]]); ax.set_ylim(recorte.total_bounds[[1, 3]])
barra_escala(ax, 500); seta_norte(ax)
titulo(ax, "Da Praça da Liberdade ao Mercado Central", f"Caminho mínimo a pé · {metros:,.0f} m")
creditos(fig, fonte="OpenStreetMap")
salvar(fig, "09_rota_a_pe")

## 12. Grafos: o sistema de transporte da RMBH como rede

| Conceito | No sistema | Em networkx |
|---|---|---|
| Nó | estação, terminal, conexão entre linhas | `G.nodes` |
| Aresta | trecho de linha entre dois nós | `G.edges` |
| Grau | quantas ligações o nó tem | `G.degree` |
| Caminho mínimo | menor trajeto entre dois nós | `nx.shortest_path` |
| Diâmetro | o maior dos caminhos mínimos (o "mais longo") | `caminho_mais_longo` |
| Betweenness | quanto o nó está "no meio" dos trajetos | `nx.betweenness_centrality` |
| Componente | pedaço da rede ainda conectado | `nx.connected_components` |

**Dados:** os 17 projetos de metrô, VLT e BRT da Região Metropolitana publicados no portal Mobilidade Brasil do BNDES (`dados/SIG_Camadas_BNDES/`), somados à Linha 1 em operação. As funções de `rede_rmbh.py` já estão prontas: aqui o foco é **ler** os resultados.

In [ ]:
from rede_rmbh import construir_rede_rmbh, CORES_MODO

G, nos, trechos, linhas = construir_rede_rmbh()
print(G.number_of_nodes(), "nós,", G.number_of_edges(), "trechos,",
      nx.number_connected_components(G), "componentes")
linhas[["sigla", "modo", "Ext_km", "Demanda"]]

### Como as linhas viraram grafo (e onde conferir)

Os arquivos do BNDES trazem **linhas**, não estações. O código cria nós nos extremos de cada linha e onde duas linhas se cruzam, funde pontos a menos de 300 m e corta cada linha nesses nós. Duas tabelas pequenas, editáveis, registram as decisões manuais:

- `dados/nos_nomeados.csv`: o nome de cada extremo, tirado do nome do projeto;
- `dados/conexoes_manuais.csv`: extremos que pertencem a uma estação da Linha 1.

Isso também é validação de dados: toda decisão fica escrita e pode ser contestada.

In [ ]:
pd.read_csv("oficina-python-mapeamento/dados/conexoes_manuais.csv")

In [ ]:
for comp in nx.connected_components(G):
    print(len(comp), "nós:", sorted(comp)[:6], "...")

O trecho **Ibirité–Barreiro** aparece isolado: o projeto que o liga ao resto da Linha 2 não está entre os 17. É uma lacuna do **recorte de dados**, não da cidade. Antes de concluir que algo está "desconectado", pergunte se o dado está completo.

In [ ]:
fig, ax = mapa_base((10, 10))
for modo, g in linhas.groupby("modo"):
    g.plot(ax=ax, color=CORES_MODO[modo], linewidth=4 if "Linha 1" in modo else 3, label=modo)
nos.plot(ax=ax, color="white", edgecolor=CORES["tinta"], markersize=18, zorder=3)
ax.legend(loc="lower left", title="Modo")
barra_escala(ax, 5000); seta_norte(ax)
titulo(ax, "Sistema estrutural de transporte da RMBH", "Linha 1 em operação e 17 projetos de metrô, VLT e BRT")
creditos(fig, fonte="BNDES, Mobilidade Brasil; Linha 1 com posição aproximada")
salvar(fig, "10_sistema_rmbh")

In [ ]:
cent = tabela_centralidades(G)
cent.head(10).round(3)

Aqui o **grau** já distingue: conexões entre linhas têm grau 3 ou 4. O **betweenness** mostra quais nós estão no meio dos trajetos. São Gabriel aparece no topo porque concentra Linha 1, BRT Cristiano Machado e BRT Anel.

In [ ]:
o, d, comp_m, cam = caminho_mais_longo(G)
print(f"Diâmetro: {o} → {d}, {comp_m / 1000:.1f} km, {len(cam) - 1} trechos")

c = nx.shortest_path(G, "Betim", "Aeroporto de Confins", weight="length")
print("Betim → Confins:", " → ".join(c))

### Fragilidade: o que acontece se um nó falha?

Retiramos, um a um, o nó mais central e recalculamos. Quanto mais rápido o maior pedaço da rede encolhe, mais frágil ela é.

In [ ]:
fragil = remocao_de_nos(G, metrica="betweenness", n=6, sequencial=True)
fragil

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.5))
ax.plot(fragil["removidos"], fragil["maior_componente"], marker="o", color=CORES["roxo"], lw=2)
for r in fragil.iloc[1:].itertuples():
    ax.annotate(r.estacao, (r.removidos, r.maior_componente), fontsize=8, xytext=(4, 6), textcoords="offset points")
ax.set_xlabel("Nós retirados (o mais central a cada passo)"); ax.set_ylabel("Nós no maior pedaço da rede")
titulo(ax, "Quanto a rede aguenta?", "Remoção sequencial por betweenness")
creditos(fig, fonte="BNDES, Mobilidade Brasil")
salvar(fig, "11_fragilidade")

### Nós e trechos mais carregados

O BNDES informa uma **demanda estimada** para cada projeto (coluna `Demanda`; confira a unidade na fonte). Quando várias linhas usam o mesmo trecho, somamos suas demandas. A **carga** de um nó é a soma da demanda das linhas que passam por ele.

A Linha 1 em operação não tem demanda nessa base, então entra com zero: é um limite que precisa aparecer na leitura.

In [ ]:
print(nos.nlargest(8, "carga")[["nome", "carga", "linhas"]].to_string(index=False))
trechos.nlargest(5, "demanda")[["de", "para", "linhas", "demanda"]]

In [ ]:
cent_nos = nos.merge(cent.reset_index(), left_on="nome", right_on="estacao")
top = cent_nos.nlargest(6, "betweenness")

fig, ax = mapa_base((10, 10))
for modo, g in trechos.groupby("modo"):
    g.plot(ax=ax, color=CORES_MODO[modo], capstyle="round",
           linewidth=1.5 + 9 * g["demanda"] / trechos["demanda"].max())
cent_nos.plot(ax=ax, color=CORES["tinta"], edgecolor="white", linewidth=0.8, zorder=3,
              markersize=15 + 260 * cent_nos["betweenness"] / cent_nos["betweenness"].max())
for r in top.itertuples():
    ax.annotate(r.nome, (r.geometry.x, r.geometry.y), xytext=(6, 4), textcoords="offset points",
                fontsize=8.5, fontweight="medium")
barra_escala(ax, 5000); seta_norte(ax)
titulo(ax, "Centralidade e carregamento", "Círculo: betweenness · espessura: demanda estimada no trecho")
creditos(fig, fonte="BNDES, Mobilidade Brasil")
salvar(fig, "12_rede_centralidade_carga")

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 6), gridspec_kw={"wspace": 0.75})
b10 = cent["betweenness"].sort_values().tail(10)
ax1.barh(b10.index, b10.values, color=CORES["roxo"])
ax1.set_title("Betweenness (top 10)", loc="left", fontsize=12)
c10 = nos.set_index("nome")["carga"].sort_values().tail(10) / 1000
ax2.barh(c10.index, c10.values, color=CORES["minerio"])
ax2.set_title("Carga do nó (mil, demanda BNDES)", loc="left", fontsize=12)
fig.suptitle("Quem segura e quem carrega a rede", x=0.125, ha="left", fontsize=22,
             fontweight="semibold", fontfamily="Barlow Condensed")
creditos(fig, fonte="BNDES, Mobilidade Brasil")
salvar(fig, "13_rede_graficos")

## 13. Compartilhar: mapa interativo e painel

Dois formatos que abrem em qualquer navegador:
1. `explore()` do geopandas, para explorar uma camada;
2. o painel da oficina: mapa do sistema da RMBH com gráficos na barra lateral, filtros por modo e simulação de remoção de nós.

In [ ]:
m = bh.explore(column="dens_ha", scheme="quantiles", k=6, cmap="felipe_seq",
               tiles="CartoDB positron", tooltip=["NM_BAIRRO", "pessoas", "dens_ha"],
               style_kwds={"weight": 0}, legend_kwds={"caption": "hab/ha"})
creditos_folium(m, fonte="IBGE, Censo 2022")
m.save("outputs/mapa_densidade_bh.html")
m

In [ ]:
exportar_painel(G, "outputs/painel_rede_rmbh.html")

## 14. Levar para casa

Os produtos estão em `outputs/` (PNG em 300 dpi e SVG editável). Para salvar no seu Drive:

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
!mkdir -p "/content/drive/MyDrive/oficina_python_mapeamento"
!cp -r outputs "/content/drive/MyDrive/oficina_python_mapeamento/"

---
**Créditos.** Oficina e código: Felipe Almeida ([felipe-almeida.com](https://felipe-almeida.com/)). Partes da análise de redes adaptadas de trabalhos desenvolvidos no MSc Urban Spatial Science (UCL CASA). Dados: IBGE (Censo 2022), IPEA (geobr), contribuidores do OpenStreetMap.